In [1]:
from pathlib import Path
import sqlite3
import json
import pandas as pd
import numpy as np

In [2]:
#Definir estructura de carpetas
BASE_DIR = Path("/content/")
DATA_RAW = BASE_DIR/"datos"/"crudos"
DATA_PROCESSED = BASE_DIR/"datos"/"procesados"
DATA_EXTERNAL = BASE_DIR/"datos"/"externos"
NOTEBOOKS = BASE_DIR/"notebooks"

print(f"Datos crudos: {DATA_RAW}")
print(f"Datos Procesados: {DATA_PROCESSED}")
print(f"Datos Externos: {DATA_EXTERNAL}")
print(f"Notebooks: {NOTEBOOKS}")

Datos crudos: /content/datos/crudos
Datos Procesados: /content/datos/procesados
Datos Externos: /content/datos/externos
Notebooks: /content/notebooks


In [ ]:
#crear segunda fuente de datos (la primera es un dataset de internet)
con = sqlite3.connect(DATA_RAW/"devops.db")
con.execute("PRAGMA foreign_keys = ON")
cursor = con.cursor()
print("Base de datos ready nigga!!!")

Base de datos ready nigga!!!


In [ ]:
#tablas
cursor.execute("""
  CREATE TABLE IF NOT EXISTS teams(
  team_id TEXT PRIMARY KEY,
  team_name TEXT NOT NULL,
  company_id INTEGER NOT NULL,
  product_area TEXT NOT NULL
  )
  """)
cursor.execute("""
  CREATE TABLE IF NOT EXISTS deployments(
  deployment_id TEXT PRIMARY KEY,
  team_id TEXT NOT NULL,
  company_id INTEGER NOT NULL,
  product_area TEXT NOT NULL,
  deployment_date TEXT NOT NULL,
  environment TEXT NOT NULL,
  status TEXT NOT NULL,
  lead_time_hours REAL NOT NULL,
  rollback_flag INTEGER NOT NULL,
  FOREIGN KEY (team_id) REFERENCES teams(team_id)
  )
""")
cursor.execute("""
CREATE TABLE IF NOT EXISTS incidents(
incident_id TEXT PRIMARY KEY,
deployment_id TEXT NOT NULL,
company_id INTEGER NOT NULL,
product_area TEXT NOT NULL,
incident_date TEXT NOT NULL,
severity TEXT NOT NULL,
resolution_time_hours REAL NOT NULL,
downtime_min INTEGER NOT NULL,
root_cause TEXT NOT NULL,
FOREIGN KEY (deployment_id) REFERENCES deployments(deployment_id)
)
""")
con.commit()
print("Tablas creadas")

Tablas creadas


In [ ]:
#Crear tercera fuente de datos (JSON)
performance_metrics = [
    {
    "metric_id": "M000001",
    "company_id": 100015,
    "product_area": "mobile",
    "timestamp": "2026-01-15T14:30:00",
    "cpu_usage_pct": 67.4,
    "memory_usage_pct": 72.1,
    "response_time_ms": 184,
    "error_rate_pct": 1.8,
    "availability_pct": 99.4,
    "requests_per_minute": 1520
}]
print("Estructura del json creada")

Estructura del json creada


In [ ]:
#Guardar el JSON
ruta_json = DATA_RAW/"performance_metrics.json"
with open(ruta_json, "w", encoding="utf-8") as archivo:
  json.dump(performance_metrics, archivo, ensure_ascii=False, indent=4)

  print(f"Archivo JSON guardado en: {ruta_json}")

Archivo JSON guardado en: /content/datos/crudos/performance_metrics.json


In [ ]:
#Verificar categorias existentes del dataset ticket que es el dataset real de internet para poder insertar datos en las demas fuentes de datos
df_tickets = pd.read_csv(DATA_RAW/"Support_tickets.csv")
df_tickets["company_id"].unique()

array([100015, 100023, 100012, 100003, 100019, 100022, 100011, 100008,
       100007, 100013, 100020, 100004, 100010, 100009, 100001, 100005,
       100025, 100006, 100017, 100002, 100024, 100018, 100014, 100016,
       100021])

In [ ]:
df_tickets["product_area"].unique()

array(['mobile', 'analytics', 'notifications', 'auth', 'billing',
       'data_pipeline'], dtype=object)

In [ ]:
df_tickets.columns.to_list()

['ticket_id',
 'day_of_week',
 'day_of_week_num',
 'company_id',
 'company_size',
 'company_size_cat',
 'industry',
 'industry_cat',
 'customer_tier',
 'customer_tier_cat',
 'org_users',
 'region',
 'region_cat',
 'past_30d_tickets',
 'past_90d_incidents',
 'product_area',
 'product_area_cat',
 'booking_channel',
 'booking_channel_cat',
 'reported_by_role',
 'reported_by_role_cat',
 'customers_affected',
 'error_rate_pct',
 'downtime_min',
 'payment_impact_flag',
 'security_incident_flag',
 'data_loss_flag',
 'has_runbook',
 'customer_sentiment',
 'customer_sentiment_cat',
 'description_length',
 'priority',
 'priority_cat']

In [ ]:
df_tickets.head()

,ticket_id,day_of_week,day_of_week_num,company_id,company_size,company_size_cat,industry,industry_cat,customer_tier,customer_tier_cat,...,downtime_min,payment_impact_flag,security_incident_flag,data_loss_flag,has_runbook,customer_sentiment,customer_sentiment_cat,description_length,priority,priority_cat
0,1000000000,Wed,3,100015,Small,1,media,7,Basic,1,...,6,0,0,0,0,neutral,2,227,low,1
1,1000000001,Sat,6,100023,Small,1,healthcare,5,Basic,1,...,2,0,0,0,0,neutral,2,461,low,1
2,1000000002,Mon,1,100012,Small,1,gaming,4,Basic,1,...,0,0,0,0,1,positive,3,306,low,1
3,1000000003,Wed,3,100003,Small,1,media,7,Plus,2,...,16,0,0,0,1,neutral,2,363,medium,2
4,1000000004,Mon,1,100019,Small,1,ecommerce,2,Plus,2,...,6,0,0,0,0,neutral,2,442,low,1


In [ ]:
#No existe un campo de fecha fok ay que crear uno para poder unir las funtes y que tengan coherencia

#Copia del dataset original
df_tickets_processed = df_tickets.copy()

#generador reproducible
rng = np.random.default_rng(42)

#crear fechas aleatorias
fechas = pd.date_range(
    start="2026-01-01",
    end="2026-06-30",
    freq="D"
)

df_tickets_processed["event_date"] = rng.choice(
    fechas,
    size=len(df_tickets_processed)
)

#convertir a formato de fehca
df_tickets_processed["event_date"] = pd.to_datetime(
    df_tickets_processed["event_date"]
).dt.date

df_tickets_processed.head()

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
df_tickets_processed.to_csv(DATA_PROCESSED/"Support_tickets_processed.csv", index=False)

In [ ]:
#Creamos la tabla "llaves" esto es importante para poder conocer las combinaciones que realmente existen en el dataset
integration_keys = (
    df_tickets_processed[["company_id","product_area","event_date"]]
    .drop_duplicates()
    .reset_index(drop=True)
)
print("Combinaciones unicas: ", len(integration_keys))
integration_keys.head()

Combinaciones unicas:  20144


,company_id,product_area,event_date
0,100015,mobile,2026-01-17
1,100023,analytics,2026-05-21
2,100012,notifications,2026-04-29
3,100003,analytics,2026-03-21
4,100019,analytics,2026-03-20


In [ ]:
from numpy.random.mtrand import f
#Generamos datos del teams (no se necitan tantos ya que un equipo puede manejar una empresa y un area determinada)
teams = (
    df_tickets_processed[["company_id","product_area"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

teams["team_id"] = [
    f"T{i:04d}" for i in range(1, len(teams) + 1)
]

teams["team_name"] = (
    teams["product_area"].str.replace("_"," ").str.title()
    + " Team"
)

teams = teams[[
    "team_id",
    "team_name",
    "company_id",
    "product_area"
]]

teams.head()

,team_id,team_name,company_id,product_area
0,T0001,Mobile Team,100015,mobile
1,T0002,Analytics Team,100023,analytics
2,T0003,Notifications Team,100012,notifications
3,T0004,Analytics Team,100003,analytics
4,T0005,Analytics Team,100019,analytics


In [ ]:
#Datos para deployment
#seleccionamos aleatoriamente las llaves

deployments = integration_keys.sample(
    n=15000,
    replace=True,
    random_state=42
).reset_index(drop=True)

In [ ]:
#Datos restantes (las llaves son sus llaves para poder unirlas) "[DAVO MEME]"
rng = np.random.default_rng(42)

deployments["deployment_id"] = [
    f"D{i:06d}" for i in range(1, len(deployments) + 1)
]

deployments["environment"] = rng.choice(
    ["staging","production"],
    size=len(deployments),
    p=[0.35,0.65]
)

deployments["status"] = rng.choice(
    ["succes", "failed"],
    size=len(deployments),
    p=[0.90,0.10]
)

deployments["lead_time_hours"] = np.round(
    rng.uniform(1,24, len(deployments)),
    2
)

deployments["rollback_flag"] = (
    deployments["status"] == "failed"
).astype(int)

In [ ]:
#Tenemos que conectar cada deployment con un equipo con un merge
deployments = deployments.merge(
    teams,
    on=["company_id","product_area"],
    how="left"
)

deployments = deployments[
    [
        "deployment_id",
        "team_id",
        "company_id",
        "product_area",
        "event_date",
        "environment",
        "status",
        "lead_time_hours",
        "rollback_flag"
    ]
]

In [ ]:
#Generamos los incidentes con una idea realista que no todos los deploys producen incidentes
incidents = deployments.sample(
    n=4000,
    random_state=42
).copy()

incidents = incidents.reset_index(drop=True)

In [ ]:
incidents["incident_id"] = [
    f"I{i:06d}" for i in range(1, len(incidents) + 1)
]

#severidad
incidents["severity"] = rng.choice(
    ["low", "medium", "high", "critical"],
    size = len(incidents),
    p=[0.30,0.40,0.22,0.08]
)

#timepos de resolucion
incidents["resolution_time_hours"] = np.round(
    rng.uniform(0.5,12,len(incidents)),
    2
)

#downtime
incidents["downtime_min"] = rng.integers(
    5,600,len(incidents)
)

#causa

incidents["root_cause"] = rng.choice([
    "code_change",
    "configuration",
    "infrastructure",
    "dependency",
    "database",
    "network"
],
size = len(incidents)
)

In [ ]:
#generamos las performance_metrics

performance = integration_keys.sample(
    n=15000,
    replace=True,
    random_state=100
).reset_index(drop=True)

In [ ]:
performance["metric_id"] = [
    f"M{i:06d}" for i in range(1, len(performance) + 1)
]

#cpu
performance["cpu_usage_pct"] = np.round(
    rng.uniform(20,95, len(performance)),
    2
)

#memoria
performance["memory_usage_pct"] = np.round(
    rng.uniform(25,90, len(performance)),
    2
)

#tiempo de respuesta
performance["response_time_ms"] = rng.integers(
    50,1000,len(performance)
)

#tasa de error
performance["error_rate_pct"] = np.round(
    rng.uniform(0.1,10,len(performance)),
    2
)

#disponibildad
performance["availability_pct"] = np.round(
    rng.uniform(95,100,len(performance)),
    3
)

#requests
performance["requests_per_minute"] = rng.integers(
    100,10000,len(performance)
)

#timestamp
performance["timestamp"] = (
    performance["event_date"].astype(str)
    + "T12:00:00"
)

In [ ]:
#guardamos el json
performance_json = performance [[
    "metric_id",
    "company_id",
    "product_area",
    "event_date",
    "timestamp",
    "cpu_usage_pct",
    "memory_usage_pct",
    "response_time_ms",
    "error_rate_pct",
    "availability_pct",
    "requests_per_minute"
]].copy()

# Convertir 'event_date' a string para que sea JSON serializable
performance_json["event_date"] = performance_json["event_date"].astype(str)

records = performance_json.to_dict(orient="records")

with open(DATA_RAW/"performance_metrics.json", "w", encoding="utf-8") as archivo:
  json.dump(records, archivo, ensure_ascii=False, indent=4)

print("JSON guardao correctamente")

JSON guardao correctamente


In [ ]:
#guardamos datos de SQLite una vez que tenemos teams, deployments e incidents
con = sqlite3.connect(DATA_RAW/"devops.db")
con.execute("PRAGMA foreign_keys = on")

teams.to_sql("teams", con, if_exists="replace",index=False)

deployments.to_sql("deployments", con, if_exists="replace", index=False)

incidents.to_sql("incidents", con, if_exists="replace", index=False)

con.commit()
con.close()
print("Datos insertados en SQLite")

Datos insertados en SQLite


In [ ]:
#verificacion de archivos
print("CSV:", (DATA_RAW / "Support_tickets.csv").exists())
print("SQLite:", (DATA_RAW / "devops.db").exists())
print("JSON:", (DATA_RAW / "performance_metrics.json").exists())

CSV: True
SQLite: True
JSON: True


In [ ]:
#verificacion de cantidad de datos en archivo CSV
print("Filas:", len(df_tickets))
print("Columnas:", len(df_tickets.columns))

df_tickets.head()

Filas: 50000
Columnas: 33


,ticket_id,day_of_week,day_of_week_num,company_id,company_size,company_size_cat,industry,industry_cat,customer_tier,customer_tier_cat,...,downtime_min,payment_impact_flag,security_incident_flag,data_loss_flag,has_runbook,customer_sentiment,customer_sentiment_cat,description_length,priority,priority_cat
0,1000000000,Wed,3,100015,Small,1,media,7,Basic,1,...,6,0,0,0,0,neutral,2,227,low,1
1,1000000001,Sat,6,100023,Small,1,healthcare,5,Basic,1,...,2,0,0,0,0,neutral,2,461,low,1
2,1000000002,Mon,1,100012,Small,1,gaming,4,Basic,1,...,0,0,0,0,1,positive,3,306,low,1
3,1000000003,Wed,3,100003,Small,1,media,7,Plus,2,...,16,0,0,0,1,neutral,2,363,medium,2
4,1000000004,Mon,1,100019,Small,1,ecommerce,2,Plus,2,...,6,0,0,0,0,neutral,2,442,low,1


In [ ]:
#Verificacion de tablas de BD
con = sqlite3.connect(DATA_RAW / "devops.db")
tablas = pd.read_sql("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name
""", con)

display(tablas)

#verificaion de datos en las tablas
for tabla in ["teams", "deployments", "incidents"]:

    resultado = pd.read_sql(
        f"SELECT COUNT(*) AS registros FROM {tabla}",
        con
    )

    print(f"{tabla}: {resultado.iloc[0,0]:,} registros")

#verificar registros reales
print()
teams_check = pd.read_sql(
    "SELECT * FROM teams LIMIT 5",
    con
)

deployments_check = pd.read_sql(
    "SELECT * FROM deployments LIMIT 5",
    con
)

incidents_check = pd.read_sql(
    "SELECT * FROM incidents LIMIT 5",
    con
)

display(teams_check)
display(deployments_check)
display(incidents_check)

con.close()

,name
0,deployments
1,incidents
2,teams


teams: 140 registros
deployments: 15,000 registros
incidents: 4,000 registros



,team_id,team_name,company_id,product_area
0,T0001,Mobile Team,100015,mobile
1,T0002,Analytics Team,100023,analytics
2,T0003,Notifications Team,100012,notifications
3,T0004,Analytics Team,100003,analytics
4,T0005,Analytics Team,100019,analytics


,deployment_id,team_id,company_id,product_area,event_date,environment,status,lead_time_hours,rollback_flag
0,D000001,T0011,100020,mobile,2026-02-26,production,succes,8.98,0
1,D000002,T0015,100015,auth,2026-02-19,production,succes,10.68,0
2,D000003,T0068,100016,auth,2026-05-26,production,succes,23.65,0
3,D000004,T0018,100009,notifications,2026-04-17,production,succes,2.94,0
4,D000005,T0086,100021,data_pipeline,2026-03-13,staging,succes,3.57,0


,deployment_id,team_id,company_id,product_area,event_date,environment,status,lead_time_hours,rollback_flag,incident_id,severity,resolution_time_hours,downtime_min,root_cause
0,D011500,T0038,100009,analytics,2026-06-08,production,succes,17.83,0,I000001,low,8.38,573,database
1,D006476,T0123,100024,analytics,2026-04-18,staging,succes,1.76,0,I000002,medium,11.07,191,configuration
2,D013168,T0071,100007,data_pipeline,2026-01-20,staging,succes,6.35,0,I000003,medium,2.14,410,database
3,D000863,T0076,100003,notifications,2026-06-10,production,succes,10.82,0,I000004,high,4.40,546,infrastructure
4,D005971,T0064,100018,data_pipeline,2026-05-22,production,failed,21.73,1,I000005,critical,4.22,50,network


In [ ]:
#verificar el JSON
with open(ruta_json, "r", encoding="utf-8") as archivo:
    performance_data = json.load(archivo)

print("Registros JSON:", len(performance_data))

df_performance = pd.DataFrame(performance_data)

print(df_performance.shape)

display(df_performance.head())

Registros JSON: 15000
(15000, 11)


,metric_id,company_id,product_area,event_date,timestamp,cpu_usage_pct,memory_usage_pct,response_time_ms,error_rate_pct,availability_pct,requests_per_minute
0,M000001,100014,mobile,2026-01-06,2026-01-06T12:00:00,91.55,32.83,81,7.47,95.759,9601
1,M000002,100001,notifications,2026-05-28,2026-05-28T12:00:00,57.20,29.39,509,4.54,99.833,3307
2,M000003,100021,notifications,2026-06-20,2026-06-20T12:00:00,61.23,78.81,471,9.37,96.033,5562
3,M000004,100002,mobile,2026-06-20,2026-06-20T12:00:00,44.65,63.64,818,5.32,98.322,1689
4,M000005,100021,notifications,2026-05-30,2026-05-30T12:00:00,44.35,70.86,912,2.34,99.970,9242


In [ ]:
#Prueba de fuego para ver si se pueden relacionar las fuentes

#llaves del csv
keys_tickets = df_tickets_processed[
    ["company_id", "product_area", "event_date"]
].drop_duplicates()

#llaves del sqlite
con = sqlite3.connect(DATA_RAW / "devops.db")
keys_deployments = pd.read_sql("""
    SELECT DISTINCT
        company_id,
        product_area,
        event_date
    FROM deployments
""", con)

keys_deployments = keys_deployments.rename(
    columns={"deployment_date": "event_date"}
)

#llaves del JSON
keys_performance = df_performance[
    ["company_id", "product_area", "event_date"]
].drop_duplicates()

In [ ]:
print(df_tickets_processed["event_date"].dtype)
print(keys_deployments["event_date"].dtype)
print(keys_performance["event_date"].dtype)

object
object
object


In [ ]:
#Prueba de combinaciones de fuentes
# Llaves de cada fuente

keys_tickets = df_tickets_processed[
    ["company_id", "product_area", "event_date"]
].drop_duplicates()

keys_deployments = pd.read_sql("""
    SELECT DISTINCT
        company_id,
        product_area,
        event_date
    FROM deployments
""", con)

keys_performance = df_performance[
    ["company_id", "product_area", "event_date"]
].drop_duplicates()


# Convertir fechas al mismo tipo

keys_tickets["event_date"] = pd.to_datetime(
    keys_tickets["event_date"]
)

keys_deployments["event_date"] = pd.to_datetime(
    keys_deployments["event_date"]
)

keys_performance["event_date"] = pd.to_datetime(
    keys_performance["event_date"]
)


# Buscar coincidencias entre las tres fuentes

keys_integradas = (
    keys_tickets
    .merge(
        keys_deployments,
        on=["company_id", "product_area", "event_date"],
        how="inner"
    )
    .merge(
        keys_performance,
        on=["company_id", "product_area", "event_date"],
        how="inner"
    )
)

print(
    "Combinaciones presentes en las tres fuentes:",
    len(keys_integradas)
)

display(keys_integradas.head(10))

Combinaciones presentes en las tres fuentes: 5477


,company_id,product_area,event_date
0,100023,analytics,2026-05-21
1,100013,billing,2026-01-18
2,100011,mobile,2026-04-06
3,100013,data_pipeline,2026-05-10
4,100004,billing,2026-05-23
5,100008,analytics,2026-06-01
6,100009,notifications,2026-04-01
7,100005,notifications,2026-04-27
8,100010,mobile,2026-05-29
9,100023,billing,2026-04-09


**Combinar las fuentes heterogeneas para poder consolidarlas**

In [4]:
#csv procesado
df_tickets = pd.read_csv(DATA_PROCESSED / "Support_tickets_processed.csv")

#SQLIite
con = sqlite3.connect(DATA_RAW/"devops.db")

#JSON
with open(DATA_RAW/"performance_metrics.json", "r", encoding="utf-8") as archivo:
    performance_data = json.load(archivo)

df_performance = pd.DataFrame(performance_data)

df_deployments = pd.read_sql(
    "SELECT * FROM deployments",
    con
)

df_incidents = pd.read_sql(
    "SELECT * FROM incidents",
    con
)

con.close()

print("Deployments:", df_deployments.shape)
print("Incidents:", df_incidents.shape)

print("Tickets: ", df_tickets.shape)
print("Performance:", df_performance.shape)

Deployments: (15000, 9)
Incidents: (4000, 14)
Tickets:  (50000, 34)
Performance: (15000, 11)


In [5]:
#Estandarizar las fechas
df_tickets["event_date"] = pd.to_datetime(
    df_tickets["event_date"]
).dt.date

df_deployments["event_date"] = pd.to_datetime(
    df_deployments["event_date"]
).dt.date

df_incidents["event_date"] = pd.to_datetime(
    df_incidents["event_date"]
).dt.date

df_performance["event_date"] = pd.to_datetime(
    df_performance["event_date"]
).dt.date

In [9]:
#tickets diarios
tickets_daily = (
    df_tickets
    .groupby(
        ["company_id", "product_area", "event_date"]
    )
    .agg(
        ticket_count=("ticket_id", "count")
    )
    .reset_index()
)

In [7]:
df_tickets.columns.tolist()

['ticket_id',
 'day_of_week',
 'day_of_week_num',
 'company_id',
 'company_size',
 'company_size_cat',
 'industry',
 'industry_cat',
 'customer_tier',
 'customer_tier_cat',
 'org_users',
 'region',
 'region_cat',
 'past_30d_tickets',
 'past_90d_incidents',
 'product_area',
 'product_area_cat',
 'booking_channel',
 'booking_channel_cat',
 'reported_by_role',
 'reported_by_role_cat',
 'customers_affected',
 'error_rate_pct',
 'downtime_min',
 'payment_impact_flag',
 'security_incident_flag',
 'data_loss_flag',
 'has_runbook',
 'customer_sentiment',
 'customer_sentiment_cat',
 'description_length',
 'priority',
 'priority_cat',
 'event_date']

In [10]:
ticket_count=("ticket_id", "count")

In [13]:
#resumir los deploys
deployments_daily = (
    df_deployments
    .groupby(
        ["company_id", "product_area", "event_date"]
    )
    .agg(
        deployment_count=("deployment_id", "count"),
        failed_deployments=("status", lambda x: (x == "failed").sum()),
        avg_lead_time_hours=("lead_time_hours", "mean"),
        rollback_count=("rollback_flag", "sum")
    )
    .reset_index()
)
print(deployments_daily.head)

<bound method NDFrame.head of        company_id   product_area  event_date  deployment_count  \
0          100001      analytics  2026-01-03                 1   
1          100001      analytics  2026-01-09                 1   
2          100001      analytics  2026-01-12                 1   
3          100001      analytics  2026-01-13                 3   
4          100001      analytics  2026-01-14                 1   
...           ...            ...         ...               ...   
10553      100025  notifications  2026-06-23                 2   
10554      100025  notifications  2026-06-24                 2   
10555      100025  notifications  2026-06-27                 1   
10556      100025  notifications  2026-06-28                 1   
10557      100025  notifications  2026-06-29                 1   

       failed_deployments  avg_lead_time_hours  rollback_count  
0                       0            18.290000               0  
1                       0            17.650000 

In [14]:
#Resumir incidentes
incidents_daily = (
    df_incidents
    .groupby(
        ["company_id", "product_area", "event_date"]
    )
    .agg(
        incident_count=("incident_id", "count"),
        avg_resolution_time_hours=(
            "resolution_time_hours",
            "mean"
        ),
        total_downtime_min=("downtime_min", "sum")
    )
    .reset_index()
)

print(incidents_daily.head)

<bound method NDFrame.head of       company_id   product_area  event_date  incident_count  \
0         100001      analytics  2026-01-14               1   
1         100001      analytics  2026-02-10               1   
2         100001      analytics  2026-02-15               1   
3         100001      analytics  2026-03-16               1   
4         100001      analytics  2026-04-02               1   
...          ...            ...         ...             ...   
3633      100025  notifications  2026-05-22               1   
3634      100025  notifications  2026-05-26               1   
3635      100025  notifications  2026-06-05               1   
3636      100025  notifications  2026-06-12               1   
3637      100025  notifications  2026-06-24               2   

      avg_resolution_time_hours  total_downtime_min  
0                          1.59                 103  
1                          8.56                 277  
2                          4.64                 111

In [15]:
#resumir performance
performance_daily = (
    df_performance
    .groupby(
        ["company_id", "product_area", "event_date"]
    )
    .agg(
        avg_cpu_usage_pct=("cpu_usage_pct", "mean"),
        avg_memory_usage_pct=("memory_usage_pct", "mean"),
        avg_response_time_ms=("response_time_ms", "mean"),
        avg_error_rate_pct=("error_rate_pct", "mean"),
        avg_availability_pct=("availability_pct", "mean"),
        avg_requests_per_minute=("requests_per_minute", "mean")
    )
    .reset_index()
)

In [16]:
#unirlos
df_integrated = tickets_daily.merge(
    deployments_daily,
    on=["company_id", "product_area", "event_date"],
    how="outer"
)

df_integrated = df_integrated.merge(
    incidents_daily,
    on=["company_id", "product_area", "event_date"],
    how="outer"
)

df_integrated = df_integrated.merge(
    performance_daily,
    on=["company_id", "product_area", "event_date"],
    how="outer"
)

In [18]:
#verificacion del df integrado
print("Filas:", len(df_integrated))
print("Columnas:", len(df_integrated.columns))

display(df_integrated.head(10))

df_integrated.info()

Filas: 20144
Columnas: 17


,company_id,product_area,event_date,ticket_count,deployment_count,failed_deployments,avg_lead_time_hours,rollback_count,incident_count,avg_resolution_time_hours,total_downtime_min,avg_cpu_usage_pct,avg_memory_usage_pct,avg_response_time_ms,avg_error_rate_pct,avg_availability_pct,avg_requests_per_minute
0,100001,analytics,2026-01-01,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,56.51,31.39,265.0,7.24,98.422,7051.0
1,100001,analytics,2026-01-02,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,100001,analytics,2026-01-03,2,1.0,0.0,18.290000,0.0,NaN,NaN,NaN,55.67,53.48,736.0,3.00,96.012,4186.0
3,100001,analytics,2026-01-04,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,65.00,75.00,313.0,5.59,95.885,854.0
4,100001,analytics,2026-01-05,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,100001,analytics,2026-01-06,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,100001,analytics,2026-01-09,2,1.0,0.0,17.650000,0.0,NaN,NaN,NaN,23.48,37.85,806.0,8.11,96.792,5324.0
7,100001,analytics,2026-01-11,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,56.99,28.87,803.0,1.24,99.349,8722.0
8,100001,analytics,2026-01-12,1,1.0,0.0,22.870000,0.0,NaN,NaN,NaN,44.79,52.72,788.0,4.36,96.455,990.0
9,100001,analytics,2026-01-13,2,3.0,0.0,4.673333,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20144 entries, 0 to 20143
Data columns (total 17 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   company_id                 20144 non-null  int64  
 1   product_area               20144 non-null  object 
 2   event_date                 20144 non-null  object 
 3   ticket_count               20144 non-null  int64  
 4   deployment_count           10558 non-null  float64
 5   failed_deployments         10558 non-null  float64
 6   avg_lead_time_hours        10558 non-null  float64
 7   rollback_count             10558 non-null  float64
 8   incident_count             3638 non-null   float64
 9   avg_resolution_time_hours  3638 non-null   float64
 10  total_downtime_min         3638 non-null   float64
 11  avg_cpu_usage_pct          10496 non-null  float64
 12  avg_memory_usage_pct       10496 non-null  float64
 13  avg_response_time_ms       10496 non-null  flo

In [19]:
#comprobar duplicados
duplicados = df_integrated.duplicated(
    subset=["company_id", "product_area", "event_date"]
).sum()

print("Duplicados en la llave de integración:", duplicados)

Duplicados en la llave de integración: 0


In [20]:
#Metricas calculadas
#tasa de fallos de deployment
df_integrated["deployment_failure_rate_pct"] = (df_integrated["failed_deployments"] /
                                                df_integrated['deployment_count']
                                                *100)

#tasa de rollback
df_integrated["rollback_rate_pct"] = (df_integrated["rollback_count"] /
                                      df_integrated["deployment_count"]
                                      *100)

In [21]:
#Guardar el csv integrado
ruta_integrado = (DATA_PROCESSED/"dataset_integrado.csv")
df_integrated.to_csv(ruta_integrado, index=False)

print(f"Dataset integrado guardado en: {ruta_integrado}")

Dataset integrado guardado en: /content/datos/procesados/dataset_integrado.csv
